# RUSA-AI Quant Training — CPU Only

Run statistical forecasting and strategy utilities in Google Colab without GPU. Outputs are saved to Google Drive. Supply a historical CSV with date and numeric value columns; demo data is not silently generated.

**Models:** ARIMA, Prophet. **Indicators:** MA5/10/30, Bollinger Bands. **Engines:** Game Theory and Hedge Bet calculators.

In [ ]:
# CELL 1 — Clone the feature branch
import os, subprocess, sys
BRANCH = 'feature/quant-training-cpu'
REPO_DIR = '/content/RusaAi'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/StrategyG/RusaAi.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)
%cd /content/RusaAi
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ.setdefault('OMP_NUM_THREADS', '2')
print('CPU-only runtime. Repository branch:', BRANCH)

In [ ]:
# CELL 2 — Install CPU quantitative dependencies
%pip -q install -r requirements-quant.txt
print('Dependencies installed.')

In [ ]:
# CELL 3 — Mount Google Drive and configure paths
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/RUSA_AI_TRAINING')
DATA_DIR = DRIVE_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = DRIVE_ROOT
CSV_PATH = DATA_DIR / 'market_data.csv'
DATE_COL = 'date'       # Change if your CSV uses another header
VALUE_COL = 'close'     # Change if your CSV uses another header
FORECAST_HORIZON = 30
TEST_SIZE = 0.20
if not CSV_PATH.exists():
    print('Upload a CSV containing historical date and numeric value columns.')
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
    if not csv_names:
        raise FileNotFoundError('No CSV uploaded. Put market_data.csv in the Drive data folder.')
    selected = csv_names[0]
    CSV_PATH.write_bytes(uploaded[selected])
print('Dataset:', CSV_PATH)
print('Output:', OUTPUT_DIR)

In [ ]:
# CELL 4 — Run training pipeline (ARIMA + Prophet + indicators)
import subprocess, sys
command = [
    sys.executable, '-m', 'rusa_quant.pipeline',
    '--data', str(CSV_PATH),
    '--date-col', DATE_COL,
    '--value-col', VALUE_COL,
    '--output', str(OUTPUT_DIR),
    '--horizon', str(FORECAST_HORIZON),
    '--test-size', str(TEST_SIZE),
    '--source-branch', BRANCH,
]
subprocess.run(command, check=True)
print('Training completed. Artifacts are on Google Drive.')

In [ ]:
# CELL 5 — Inspect evaluation and forecast outputs
import pandas as pd, json
metrics_path = DRIVE_ROOT / 'reports' / 'model_metrics.csv'
forecast_path = DRIVE_ROOT / 'results' / 'future_forecasts.csv'
manifest_path = DRIVE_ROOT / 'reports' / 'training_manifest.json'
display(pd.read_csv(metrics_path))
display(pd.read_csv(forecast_path).head(10))
manifest = json.loads(manifest_path.read_text())
print(json.dumps({k: manifest[k] for k in ['execution_device', 'dataset_rows', 'selected_arima_order', 'metrics']}, indent=2))

In [ ]:
# CELL 6 — Optional Game Theory and Hedge Bet calculators
# These are payoff/odds calculators, not trained predictors. Replace demo inputs with your own assumptions.
from rusa_quant.engines.game_theory import game_theory_engine
from rusa_quant.engines.hedge_bet import hedge_two_outcomes
import json
RUN_EXAMPLE_CALCULATIONS = False
if RUN_EXAMPLE_CALCULATIONS:
    game_result = game_theory_engine([[1, -1], [-1, 1]])
    # Example odds only; this does not imply a recommendation or guaranteed profit.
    hedge_result = hedge_two_outcomes(2.10, 1.90, 100000)
    (DRIVE_ROOT / 'reports' / 'game_theory_result.json').write_text(json.dumps(game_result, indent=2))
    (DRIVE_ROOT / 'reports' / 'hedge_calculation.json').write_text(json.dumps(hedge_result, indent=2))
    print('Strategy calculations saved to Drive.')
else:
    print('Skipped optional calculators. Set RUN_EXAMPLE_CALCULATIONS=True to run the examples.')

In [ ]:
# CELL 7 — Optional: publish sanitized training manifest to GitHub branch
# Configure a Colab Secret named RUSA_GITHUB_TOKEN with Contents read/write on this repo.
# Only the run manifest is published; no dataset, model weights, or forecast CSV is uploaded.
from datetime import datetime, timezone
import base64, requests
try:
    from google.colab import userdata
    github_token = userdata.get('RUSA_GITHUB_TOKEN')
except Exception:
    github_token = None
if github_token:
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    manifest_text = manifest_path.read_text(encoding='utf-8')
    api_url = 'https://api.github.com/repos/StrategyG/RusaAi/contents/training-metadata/' + run_id + '.json'
    headers = {'Authorization': 'Bearer ' + github_token, 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}
    payload = {'message': 'chore(quant): publish sanitized training manifest ' + run_id,
               'content': base64.b64encode(manifest_text.encode('utf-8')).decode('ascii'),
               'branch': BRANCH}
    response = requests.put(api_url, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    print('Manifest published:', response.json()['content']['path'])
else:
    print('GitHub metadata publish skipped. Drive results are saved; add RUSA_GITHUB_TOKEN Colab Secret to enable GitHub publishing.')